# L2p8_m9 multi-lightcone Poisson check: binned $N(M_{500c}, z)$

Load the eight FLAMINGO L2p8_m9 halo-lightcone catalogues ($M_{500c}\ge10^{13}\,M_\odot$,
$0\le z\le 3$) from `/rds/rds-lxu/flamingo/L2p8_m9/lightcone{0..7}/catalogues/` and
build a common $(M_{500c}, z)$ histogram for each observer.

For every bin we compare the **mean** and **sample std** of the eight observer counts to the
Poisson expectation $\sigma_N = \sqrt{\langle N\rangle}$.

Plots:
1. $N(M_{500c})$ in selected redshift slices (marginal over $z$ within each slice).
2. $N(z)$ marginal over mass.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "axes.linewidth": 0.8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
})

BASE = Path("/rds/rds-lxu/flamingo/L2p8_m9")
CAT_NAME = "halo_catalogue_M500c_1e13_zlt3_L2p8_m9_yang26rot.csv"
N_OBS = 8
CHUNK = 2_000_000

# Mass and redshift bin edges (M_500c in Msun)
Z_EDGES = np.linspace(0.0, 3.0, 16)          # 15 z bins
M_EDGES = np.logspace(13.0, 15.2, 13)        # 12 log-spaced M500c bins
Z_SLICE_IDX = [2, 5, 8, 11]                  # z-bin indices for N(M) panels


def catalogue_path(obs: int) -> Path:
    return BASE / f"lightcone{obs}" / "catalogues" / CAT_NAME


def binned_Mz_counts(path: Path) -> np.ndarray:
    """Return (n_z, n_m) integer counts from a single catalogue."""
    counts = np.zeros((len(Z_EDGES) - 1, len(M_EDGES) - 1), dtype=np.int64)
    reader = pd.read_csv(
        path,
        comment="#",
        usecols=["z", "M_500c_Msun"],
        chunksize=CHUNK,
    )
    for chunk in reader:
        z = chunk["z"].to_numpy(dtype=np.float64)
        m = chunk["M_500c_Msun"].to_numpy(dtype=np.float64)
        keep = np.isfinite(z) & np.isfinite(m) & (z >= Z_EDGES[0]) & (z <= Z_EDGES[-1])
        if not keep.any():
            continue
        h, _, _ = np.histogram2d(
            z[keep], m[keep], bins=[Z_EDGES, M_EDGES]
        )
        counts += h.astype(np.int64)
    return counts


paths = [catalogue_path(obs) for obs in range(N_OBS)]
for p in paths:
    if not p.exists():
        raise FileNotFoundError(p)

counts_lc = np.stack([binned_Mz_counts(p) for p in paths], axis=0)  # (8, n_z, n_m)
print(f"loaded {N_OBS} lightcones | grid {counts_lc.shape[1]} z x {counts_lc.shape[2]} M bins")
print(f"total rows (sum over bins/observers): {counts_lc.sum():,}")

## Helper: mean, std, and Poisson reference

In [ ]:
def observer_stats(counts_1d: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """counts_1d shape (n_obs, n_bins)."""
    mean = counts_1d.mean(axis=0)
    std = counts_1d.std(axis=0, ddof=1)
    poisson = np.sqrt(np.maximum(mean, 0.0))
    return mean, std, poisson


M_CENT = 0.5 * (M_EDGES[:-1] + M_EDGES[1:])
Z_CENT = 0.5 * (Z_EDGES[:-1] + Z_EDGES[1:])

## $N(M_{500c})$ in selected $z$ slices

For each redshift slice we sum the $(M,z)$ grid over $z$ within the slice, then compare the
eight observer counts in each mass bin.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True)
axes = axes.ravel()

for ax, zi in zip(axes, Z_SLICE_IDX):
    z_lo, z_hi = Z_EDGES[zi], Z_EDGES[zi + 1]
    # one z bin only (not cumulative over multiple bins)
    n_m_obs = counts_lc[:, zi, :]  # (8, n_m)
    mean, std, poisson = observer_stats(n_m_obs)

    mask = mean > 0
    ax.errorbar(
        M_CENT[mask], mean[mask], yerr=std[mask],
        fmt="o", ms=4, capsize=2, color="C0", label=r"mean $\pm$ std (8 lc)",
    )
    ax.plot(
        M_CENT[mask], poisson[mask],
        "--", color="C1", lw=1.5, label=r"Poisson $\sqrt{\langle N\rangle}$",
    )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(rf"$z \in [{z_lo:.2f},\,{z_hi:.2f}]$")
    ax.set_ylabel(r"$N(M_{500c})$")
    ax.grid(True, which="both", alpha=0.25)

axes[-1].set_xlabel(r"$M_{500c}$ [$M_\odot$]")
axes[0].legend(loc="upper right", frameon=True)
fig.suptitle(
    r"L2p8_m9: $N(M_{500c})$ per lightcone vs Poisson scatter (8 observers)",
    y=1.02,
)
fig.tight_layout()
plt.show()

## Ratio $\sigma_{\rm obs} / \sqrt{\langle N\rangle}$ for $N(M_{500c}|z)$

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True)
axes = axes.ravel()

for ax, zi in zip(axes, Z_SLICE_IDX):
    z_lo, z_hi = Z_EDGES[zi], Z_EDGES[zi + 1]
    mean, std, poisson = observer_stats(counts_lc[:, zi, :])
    ratio = np.full_like(mean, np.nan)
    ok = poisson > 0
    ratio[ok] = std[ok] / poisson[ok]

    ax.axhline(1.0, color="k", ls="--", lw=1)
    ax.plot(M_CENT[ok], ratio[ok], "o", ms=4, color="C0")
    ax.set_xscale("log")
    ax.set_ylim(0, 2.5)
    ax.set_title(rf"$z \in [{z_lo:.2f},\,{z_hi:.2f}]$")
    ax.set_ylabel(r"$\sigma_{\rm obs} / \sqrt{\langle N\rangle}$")
    ax.grid(True, alpha=0.25)

axes[-1].set_xlabel(r"$M_{500c}$ [$M_\odot$]")
fig.suptitle(r"Poisson consistency: observer std vs $\sqrt{\langle N\rangle}$", y=1.02)
fig.tight_layout()
plt.show()

## $N(z)$ marginal over $M_{500c}$

In [ ]:
n_z_obs = counts_lc.sum(axis=2)  # (8, n_z)
mean_z, std_z, poisson_z = observer_stats(n_z_obs)

fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(8, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]})

mask = mean_z > 0
ax0.errorbar(
    Z_CENT[mask], mean_z[mask], yerr=std_z[mask],
    fmt="o", ms=4, capsize=2, color="C0", label=r"mean $\pm$ std (8 lc)",
)
ax0.plot(
    Z_CENT[mask], poisson_z[mask],
    "--", color="C1", lw=1.5, label=r"Poisson $\sqrt{\langle N\rangle}$",
)
ax0.set_yscale("log")
ax0.set_ylabel(r"$N(z)$")
ax0.legend(loc="upper right")
ax0.grid(True, which="both", alpha=0.25)
ax0.set_title(r"L2p8_m9: $N(z)$ summed over $M_{500c}$ bins (8 observers)")

ratio_z = np.full_like(mean_z, np.nan)
ok = poisson_z > 0
ratio_z[ok] = std_z[ok] / poisson_z[ok]
ax1.axhline(1.0, color="k", ls="--", lw=1)
ax1.plot(Z_CENT[ok], ratio_z[ok], "o", ms=4, color="C0")
ax1.set_ylim(0, 2.5)
ax1.set_xlabel(r"$z$")
ax1.set_ylabel(r"$\sigma_{\rm obs} / \sqrt{\langle N\rangle}$")
ax1.grid(True, alpha=0.25)

fig.tight_layout()
plt.show()

## Full $(M_{500c}, z)$ grid: mean count and Poisson ratio

Heat map of $\langle N\rangle$ across observers and the ratio map
$\sigma_{\rm obs}/\sqrt{\langle N\rangle}$.

In [ ]:
mean_2d = counts_lc.mean(axis=0)
std_2d = counts_lc.std(axis=0, ddof=1)
poisson_2d = np.sqrt(np.maximum(mean_2d, 0.0))
ratio_2d = np.divide(std_2d, poisson_2d, out=np.full_like(std_2d, np.nan), where=poisson_2d > 0)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 4.5))

im0 = ax0.pcolormesh(Z_EDGES, M_EDGES, mean_2d.T, norm=plt.matplotlib.colors.LogNorm())
ax0.set_yscale("log")
ax0.set_xlabel(r"$z$")
ax0.set_ylabel(r"$M_{500c}$ [$M_\odot$]")
ax0.set_title(r"$\langle N\rangle$ over 8 lightcones")
fig.colorbar(im0, ax=ax0, pad=0.02)

im1 = ax1.pcolormesh(Z_EDGES, M_EDGES, ratio_2d.T, vmin=0, vmax=2)
ax1.set_yscale("log")
ax1.set_xlabel(r"$z$")
ax1.set_ylabel(r"$M_{500c}$ [$M_\odot$]")
ax1.set_title(r"$\sigma_{\rm obs}/\sqrt{\langle N\rangle}$")
fig.colorbar(im1, ax=ax1, pad=0.02)

fig.tight_layout()
plt.show()